In [11]:
import pandas as pd
import random
import re
import os
import json
from tqdm import tqdm
import hashlib

import json
import time

# use claude batch api to take row_content and return a "canonical_marker"
from anthropic import Anthropic, transform_schema
from anthropic.types.beta.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.beta.messages.batch_create_params import Request
import dotenv
dotenv.load_dotenv()

from pydantic import BaseModel, Field
from enum import Enum


In [2]:
# list files in /Users/ataylor/Downloads/ch_metadata_agent/htan_data


directory = '/Users/ataylor/Downloads/ch_metadata_agent/htan_data'
files = [f for f in os.listdir(directory)]
print(len(files))

716


In [7]:
# Based on the deduplication above write a function to parse each tsv or csv in the dict 
# into json with the file hash as the key

# the json should be flat and therefore have
# row_hash, row_content as dict, and file_paths as list
# and be deduiplicated based on row content

# we can also drp these keys
# "Component": "ImagingLevel2", "Filename": "mxif_level_2/AFsubtracted/HTA11_3252_20000010115220260050000000000.tif", "File Format": "tif", "HTAN Participant ID": "HTA11_3252", "HTAN Parent Biospecimen ID": "HTA11_3252_2000001011", "HTAN Data File ID": "HTA11_3252_20000010115220260050000000000", "Channel Metadata Filename": "mxif_level_2/AFsubtracted/metadata/HTA11_3252_20000010115220260050000000000_metadata.csv", "Imaging Assay Type": "MxIF", "Protocol Link": "NONE", "Workflow Start Datetime": "09/29/2020", "Workflow End Datetime": "09/30/2020", "Software and Version": "ImageApp 1.0.0.0", "Microscope": "INCELL ANALYZER 2500 HS", "Objective": "NIKON MRD00205 Plan Apochromat", "NominalMagnification": "20X", "LensNA": 0.75, "WorkingDistance": 1, "WorkingDistanceUnit": "mm", "Immersion": "Air", "Pyramid": "No", "Zstack": "Yes", "Tseries": "No", "Passed QC": "Yes", "Comment": NaN, "FOV number": NaN, "FOVX": NaN, "FOVXUnit": NaN, "FOVY": NaN, "FOVYUnit": NaN, "Frame Averaging": NaN, "Image ID": "MAP03252_0000_06_04_005\\AFR\\MAP03252_0000_06_04_005_ERBB2_AFR.tif", "DimensionOrder": "XYCZT", "PhysicalSizeX": 0.325, "PhysicalSizeXUnit": "µm", "PhysicalSizeY": 0.325, "PhysicalSizeYUnit": "µm", "PhysicalSizeZ": 0, "PhysicalSizeZUnit": "µm", "Pixels BigEndian": false, "PlaneCount": 0, "SizeC": 26, "SizeT": 0, "SizeX": 9375, "SizeY": 9402, "SizeZ": 0, "PixelType": "uint16", "LEVEL": "2-Processed", "TYPE": "3-AFsubtracted", "LAYERS": 26, "REGION": 5, "POSITION": 0, "LAYER": 12, "ROUND": 17, 


def parse_metadata_to_json(directory):
    result = []
    seen_rows = {}

    for file in files:
        if file.endswith('.tsv') or file.endswith('.csv'):
            file_path = os.path.join(directory, file)
            # regex match syn\d+ to get file_synid
            file_synid = None
            match = re.search(r'syn\d+', file)
            if match:
                file_synid = match.group(0)
            
            try:
                if file.endswith('.tsv'):
                    df = pd.read_csv(file_path, sep='\t', encoding='utf-8')
                else:
                    df = pd.read_csv(file_path, encoding='utf-8')
            except UnicodeDecodeError:
                try:
                    if file.endswith('.tsv'):
                        df = pd.read_csv(file_path, sep='\t', encoding='latin1')
                    else:
                        df = pd.read_csv(file_path, encoding='latin1')
                except Exception as e:
                    print(f"Error reading {file_path}: {e}")
                    continue

            for _, row in df.iterrows():
                row_content = row.to_dict()
                # drop the keys we don't need see above
                keys_to_drop = [
                    "Component",
                    "Filename",
                    "File Format", 
                    "HTAN Participant ID", 
                    "HTAN Parent Biospecimen ID",
                    "HTAN Data File ID", 
                    "Channel Metadata Filename", 
                    "Imaging Assay Type", 
                    "Protocol Link", 
                    "Workflow Start Datetime",
                      "Workflow End Datetime", 
                      "Software and Version", 
                      "Microscope", 
                      "Objective", 
                      "NominalMagnification", 
                      "LensNA", 
                      "WorkingDistance",
                      "WorkingDistanceUnit",
                      "Pyramid","Zstack", "Tseries", "Passed QC", "Comment", "FOV number", 
                      "FOVX", "FOVXUnit", "FOVY", "FOVYUnit", "Frame Averaging", 
                      "Image ID",
                      "DimensionOrder", "PhysicalSizeX", "PhysicalSizeXUnit", 
                      "PhysicalSizeY", 
                      "PhysicalSizeYUnit", 
                      "PhysicalSizeZ", 
                      "PhysicalSizeZUnit",
                      "Pixels BigEndian", 
                      "PlaneCount",
                        "SizeC", "SizeT", "SizeX", "SizeY", "SizeZ", "PixelType", 
                        "LEVEL", "TYPE", "LAYERS", 
                        "REGION", "POSITION", "LAYER", "ROUND"] 
                
                for key in keys_to_drop:
                    row_content.pop(key, None)

                row_str = json.dumps(row_content, sort_keys=True)
                row_hash = hashlib.md5(row_str.encode('utf-8')).hexdigest()

                if row_hash in seen_rows:
                    # Add file path to existing entry
                    seen_rows[row_hash]['ch_synids'].append(file_synid)
                else:
                    # Create new entry
                    entry = {
                        'row_hash': row_hash,
                        'row_content': row_content,
                        'ch_synids': [file_synid]
                    }
                    seen_rows[row_hash] = entry
                    result.append(entry)

    return result

metadata_json = parse_metadata_to_json(directory)

print(f"Parsed {len(metadata_json)} unique rows from metadata files into JSON format.")


Parsed 3647 unique rows from metadata files into JSON format.


In [8]:
# sample 5 random entries
metadata_json_small = random.sample(metadata_json, 5)
print(json.dumps(metadata_json_small, indent=2))

[
  {
    "row_hash": "0c268a599ded8bd0ad8b1daacdd340fb",
    "row_content": {
      "filename": "KB_SMMART_976_D23_C01R2_PD1",
      "Channel ID": NaN,
      "Channel Name": "PD1",
      "Cycle Number": 1,
      "Sub-Cycle #": 2,
      "Antibody name": "anti PD-1",
      "Clone": "NAT105",
      "Vendor": "Abcam",
      "Catalog #": "ab52587",
      "QC comment": "nonspecific background"
    },
    "ch_synids": [
      "syn69046896"
    ]
  },
  {
    "row_hash": "308d5658c7fc509ef7c19cb046e5ba54",
    "row_content": {
      "Channel ID": 34,
      "Cycle Number": 9,
      "Channel Name": "Blank_488",
      "Flurophore": "FITC",
      "Excitation Wavelength": 490,
      "Emission Wavelength": 525
    },
    "ch_synids": [
      "syn26486786",
      "syn26486778",
      "syn26486750",
      "syn26486746",
      "syn26486752",
      "syn26486753",
      "syn26486780",
      "syn26486756",
      "syn26486770",
      "syn26486758",
      "syn26486772",
      "syn26486788"
    ]
  },
  {
 

In [63]:


# marker type enum can be chemical_stain, blank_or_background, protein_group, protein_single, cd_marker, other_dna, other

class MarkerTypeEnum(str, Enum):
    chemical_stain = "chemical_stain"
    blank_or_background = "blank_or_background" 
    protein_group = "protein_group"
    protein_single = "protein_single"
    chemical_element = "chemical_element"
    cd_marker = "cd_marker"
    nuclear_marker = "nuclear_marker"
    other = "other"

class ExtractedlMarkerSchema(BaseModel):
    extracted_marker: str = Field(..., description="The most relevant canonical marker or identifier for the protein, stain or feature being imaged extracted from the metadata row content.")
    marker_type: MarkerTypeEnum = Field(..., description="The type of the canonical marker.")

client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

In [64]:
# make a single request to test
def get_marker_classification(entry):
    prompt = f"""
    You are an expert bio-imaging data curator. Your task is to analyze the metadata row content below and extract the canonical target (marker) being imaged.

    Metadata Row Content: {json.dumps(entry['row_content'])}

    ### Classification Rules
    Analyze the extracted value and categorize it into exactly one of the following types:

    1. **cd_marker**: STRICTLY for Cluster of Differentiation markers starting with "CD" (e.g., "CD3", "CD45RO", "CD8a").
    2. **chemical_stain**: Common histological that are not strictly markers.
    3. **protein_single**: A specific non-CD protein target (e.g., "Ki67", "FoxP3", "Keratin", "Beta-Catenin").
    4. **protein_group**: A broad family of proteins without a specific isoform (e.g., "Caspases", "Histones").
    5. **chemical_element**: Elemental isotopes often used in mass cytometry/IMC (e.g., "191Ir", "Ca40", "Na23") if they are the primary focus.
    6. **nuclear_marker**: Simnpe nuclear stains like DAPI, Hoechst, or general mentions of DNA, dsDNA, nucleus etc.
    7. **blank_or_background**: Use this if the row indicates a control, empty channel, "Empty", "Blank", or pure background/autofluorescence.
    8. **other**: Anything that does not fit the above.

    ### Extraction Guidelines
    - Clean the marker name (remove prefixes like "Anti-" or suffixes like " (FITC)" unless necessary).
    - If the content mentions a metal tag AND a protein (e.g., "173Yb_CD3"), extracted_marker should be "CD3" and type should be "cd_marker".

    Return the result as a valid JSON object matching the requested schema.
    """
    response = client.beta.messages.create(
        model="claude-haiku-4-5",
        max_tokens=100,
        temperature=0.0,
        betas=["structured-outputs-2025-11-13"],
        messages=[{
            "role": "user",
            "content": prompt
        }],
        output_format={
            "type": "json_schema",
            "schema": transform_schema(ExtractedlMarkerSchema)
        }
    )
    return response.content[0].text

get_marker_classification(metadata_json_small[0])

'{"extracted_marker": "PD1", "marker_type": "protein_single"}'

In [ ]:


client = Anthropic()

def process_all_metadata_with_batch(entries):
    # ... [Batch creation and submission logic same as before] ...
    
    # 1. Create Requests
    requests = []
    for idx, entry in enumerate(entries):

        prompt = f"""
        You are an expert bio-imaging data curator. Your task is to analyze the metadata row content below and extract the canonical target (marker) being imaged.

        Metadata Row Content: {json.dumps(entry['row_content'])}

        ### Classification Rules
        Analyze the extracted value and categorize it into exactly one of the following types:

        1. **cd_marker**: STRICTLY for Cluster of Differentiation markers starting with "CD" (e.g., "CD3", "CD45RO", "CD8a").
        2. **chemical_stain**: Common histological that are not strictly markers.
        3. **protein_single**: A specific non-CD protein target (e.g., "Ki67", "FoxP3", "Keratin", "Beta-Catenin").
        4. **protein_group**: A broad family of proteins without a specific isoform (e.g., "Caspases", "Histones").
        5. **chemical_element**: Elemental isotopes often used in mass cytometry/IMC (e.g., "191Ir", "Ca40", "Na23") if they are the primary focus.
        6. **nuclear_marker**: Simnpe nuclear stains like DAPI, Hoechst, or general mentions of DNA, dsDNA, nucleus etc.
        7. **blank_or_background**: Use this if the row indicates a control, empty channel, "Empty", "Blank", or pure background/autofluorescence.
        8. **other**: Anything that does not fit the above.

        ### Extraction Guidelines
        - Clean the marker name (remove prefixes like "Anti-" or suffixes like " (FITC)" unless necessary).
        - If the content mentions a metal tag AND a protein (e.g., "173Yb_CD3"), extracted_marker should be "CD3" and type should be "cd_marker".

        Return the result as a valid JSON object matching the requested schema.
    """

        requests.append({
            "custom_id": str(idx),
            "params": {
                "model": "claude-haiku-4-5",
                "max_tokens": 100,
                "messages": [{"role": "user", "content": prompt}],
                "output_format": {
                    "type": "json_schema", 
                    "schema": transform_schema(ExtractedlMarkerSchema)
                }
            }
        })

    # 2. Submit Batch
    batch = client.beta.messages.batches.create(
        requests=requests,
        betas=["structured-outputs-2025-11-13"]
    )
    print(f"Batch {batch.id} submitted.")

    # 3. Wait for Completion
    while True:
        batch = client.beta.messages.batches.retrieve(batch.id)

        print(batch.processing_status)
        if batch.processing_status == "ended":
            break
        time.sleep(10)

    # 4. Map Results and Capture Specific Errors
    results_map = {res.custom_id: res for res in client.beta.messages.batches.results(batch.id)}
    
    enhanced_entries = []
    
    for idx, entry in enumerate(entries):
        entry_enhanced = entry.copy()
        result_obj = results_map.get(str(idx))

        if not result_obj:
            # Case 1: The ID is completely missing from results (rare)
            entry_enhanced['marker_classification'] = {
                "error": "Batch result missing for this ID"
            }
        
        elif result_obj.result.type == "succeeded":
            # Case 2: Success - Parse the JSON content
            try:
                raw_json = result_obj.result.message.content[0].text
                entry_enhanced['marker_classification'] = json.loads(raw_json)
            except json.JSONDecodeError:
                entry_enhanced['marker_classification'] = {
                    "error": "Model returned invalid JSON",
                    "raw_output": raw_json
                }

        elif result_obj.result.type == "errored":
            # Case 3: API Error - Capture the REAL error message
            # The 'error' object typically has 'type' and 'message' fields
            api_error = result_obj.result.error
            error_message = getattr(api_error, "message", str(api_error))
            error_type = getattr(api_error, "type", "unknown_error")
            
            entry_enhanced['marker_classification'] = {
                "error": error_message,
                "error_type": error_type
            }

        else:
            # Case 4: Expired or other status
            entry_enhanced['marker_classification'] = {
                "error": f"Unexpected status: {result_obj.result.type}"
            }
            
        enhanced_entries.append(entry_enhanced)

    return enhanced_entries

enhanced_metadata = process_all_metadata_with_batch(metadata_json)

Batch msgbatch_01GSXeHkXesUWGV8TwbxUSxw submitted.
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
in_progress
ended


In [43]:
# save enhanced metadata to json file
with open('classified_metadata.json', 'w') as f:
    json.dump(enhanced_metadata, f, indent=2)

In [44]:
classified_df = pd.DataFrame(enhanced_metadata)
classified_df.head()

,row_hash,row_content,ch_synids,marker_classification
0,82e7ec56844f86400176e4822aa10926,"{'Immersion': 'Air', 'CHANNEL': 'UV', 'MARKERI...","[syn25126380, syn25126394, syn25126425, syn251...","{'extracted_marker': 'DAPI', 'marker_type': 'c..."
1,dbe50780b9362ce4c1c700fbab5cdfed,"{'Immersion': 'Air', 'CHANNEL': 'GFP', 'MARKER...","[syn25126380, syn25126394, syn25126425, syn251...","{'extracted_marker': 'Alpha-actinin 4', 'marke..."
2,45803a4b40c470cccfd0f6e7d2b9090e,"{'Immersion': 'Air', 'CHANNEL': 'Cy3', 'MARKER...","[syn25126380, syn25126394, syn25126425, syn251...","{'extracted_marker': 'Beta-Catenin', 'marker_t..."
3,3beeed44291c4a55cfa346f4ad59fffd,"{'Immersion': 'Air', 'CHANNEL': 'GFP', 'MARKER...","[syn25126380, syn25126394, syn25126425, syn251...","{'extracted_marker': 'CD11B', 'marker_type': '..."
4,ce7741fc0da0adb15b4a0fb5fa947ae5,"{'Immersion': 'Air', 'CHANNEL': 'Cy5', 'MARKER...","[syn25126380, syn25126394, syn25126425, syn251...","{'extracted_marker': 'CD20', 'marker_type': 'c..."


In [45]:
classified_df = pd.DataFrame(enhanced_metadata)
classified_df['extracted_marker'] = classified_df['marker_classification'].apply(lambda x: x.get('extracted_marker') if isinstance(x, dict) and 'extracted_marker' in x else None)
classified_df['marker_type'] = classified_df['marker_classification'].apply(lambda x: x.get('marker_type') if isinstance(x, dict) and 'marker_type' in x else None)
classified_df.head()

,row_hash,row_content,ch_synids,marker_classification,extracted_marker,marker_type
0,82e7ec56844f86400176e4822aa10926,"{'Immersion': 'Air', 'CHANNEL': 'UV', 'MARKERI...","[syn25126380, syn25126394, syn25126425, syn251...","{'extracted_marker': 'DAPI', 'marker_type': 'c...",DAPI,chemical_stain
1,dbe50780b9362ce4c1c700fbab5cdfed,"{'Immersion': 'Air', 'CHANNEL': 'GFP', 'MARKER...","[syn25126380, syn25126394, syn25126425, syn251...","{'extracted_marker': 'Alpha-actinin 4', 'marke...",Alpha-actinin 4,protein_single
2,45803a4b40c470cccfd0f6e7d2b9090e,"{'Immersion': 'Air', 'CHANNEL': 'Cy3', 'MARKER...","[syn25126380, syn25126394, syn25126425, syn251...","{'extracted_marker': 'Beta-Catenin', 'marker_t...",Beta-Catenin,protein_single
3,3beeed44291c4a55cfa346f4ad59fffd,"{'Immersion': 'Air', 'CHANNEL': 'GFP', 'MARKER...","[syn25126380, syn25126394, syn25126425, syn251...","{'extracted_marker': 'CD11B', 'marker_type': '...",CD11B,cd_marker
4,ce7741fc0da0adb15b4a0fb5fa947ae5,"{'Immersion': 'Air', 'CHANNEL': 'Cy5', 'MARKER...","[syn25126380, syn25126394, syn25126425, syn251...","{'extracted_marker': 'CD20', 'marker_type': 'c...",CD20,cd_marker


In [59]:
import json

def group_by_extracted_marker(enhanced_metadata):
    """
    Restructures metadata to be centered around the extracted_marker.
    Deduplicates based on a normalized version of the marker string.
    """
    classified_marker_map = {}

    for entry in enhanced_metadata:
        marker_info = entry.get('marker_classification', {})
        
        # Ensure marker_info is a dict and has the required key
        if not isinstance(marker_info, dict) or 'extracted_marker' not in marker_info:
            continue
            
        extracted_marker = marker_info['extracted_marker']
        marker_type = marker_info.get('marker_type', 'Unknown')

        # Safeguard: Ensure extracted_marker is a string before processing
        if not isinstance(extracted_marker, str):
            continue

        # Normalize: lowercase and alphanumeric only for deduplication
        normalized_marker = ''.join(c.lower() for c in extracted_marker if c.isalnum())
        
        # Skip empty markers after normalization
        if not normalized_marker:
            continue

        if normalized_marker not in classified_marker_map:
            classified_marker_map[normalized_marker] = {
                'extracted_marker': extracted_marker,  # Preserves the casing of the first occurrence
                'marker_type': marker_type,            # Preserves the type of the first occurrence
                'entries': []
            }
        
        # Add the current entry to the list for this marker
        classified_marker_map[normalized_marker]['entries'].append(entry)

    # Convert the map values to a list
    classified_marker_list = list(classified_marker_map.values())
    
    return classified_marker_list

# --- Usage ---

# Assuming 'enhanced_metadata' is already loaded in your environment
classified_markers = group_by_extracted_marker(enhanced_metadata)

print(f"Number of unique extracted markers: {len(classified_markers)}")

# Save to JSON
with open('classified_markers.json', 'w') as f:
    json.dump(classified_markers, f, indent=2)

Number of unique extracted markers: 642


In [27]:
# classify protein_single markers with uniprot search

from uniprot_api import lookup_protein

classified_protein_single = []
for marker_entry in classified_marker_list:
    if marker_entry['marker_type'] == 'protein_single':
        extracted_marker = marker_entry['extracted_marker']
        # search uniprot
        search_results = lookup_protein(extracted_marker)
        # analyze results to find best match
        # for simplicity, just take the first result here
        if search_results:
            best_match = search_results[0]
            marker_entry['uniprot_accession'] = best_match.accession
            marker_entry['gene_name'] = best_match.gene_name
            marker_entry['confidence_level'] = 'medium'  # placeholder
            marker_entry['selection_reasoning'] = 'First match from UniProt search'  # placeholder
        else:
            marker_entry['uniprot_accession'] = None
            marker_entry['gene_name'] = None
            marker_entry['confidence_level'] = 'low'
            marker_entry['selection_reasoning'] = 'No matches found in UniProt'
        
        classified_protein_single.append(marker_entry)

print(json.dumps(classified_protein_single, indent=2))

[
  {
    "extracted_marker": "Alpha-actinin 4",
    "marker_type": "protein_single",
    "entries": [
      {
        "row_hash": "dbe50780b9362ce4c1c700fbab5cdfed",
        "row_content": {
          "Immersion": "Air",
          "CHANNEL": "GFP",
          "MARKERID": 22,
          "MARKERNAME": "Alpha-actinin 4",
          "AbID": NaN,
          "CLONE": "EPR2533(2)",
          "DYE": "A488",
          "VENDOR": "Abcam",
          "CATNUM": "ab198608",
          "EXPOSURE(ms)": 100,
          "DILUTION": 0.11111
        },
        "ch_synids": [
          "syn25126380",
          "syn25126394",
          "syn25126425",
          "syn25126431",
          "syn25126419",
          "syn25126418",
          "syn25126430",
          "syn25126424",
          "syn25126395",
          "syn25126381",
          "syn25126397",
          "syn25126383",
          "syn25126432",
          "syn25126426",
          "syn25126368",
          "syn25126369",
          "syn25126427",
          "syn25126

In [53]:
# for cd_marker use cd_molecules.csv to lookup markers
cd_molecules_df = pd.read_csv('cd_molecules.csv')

classified_marker_list_df = pd.DataFrame(classified_marker_list)
cd_df = classified_marker_list_df[classified_marker_list_df['marker_type'] == 'cd_marker']
# add a column which is description from cd_molecules_df
# will need to match on upper case extracted_marker and upper case cd_name
cd_df = cd_df.merge(cd_molecules_df, left_on=cd_df['extracted_marker'].str.upper(), right_on=cd_molecules_df['cd_name'].str.upper(), how='left')

cd_df.head()

,key_0,extracted_marker,marker_type,entries,cd_name,description
0,CD11B,CD11B,cd_marker,[{'row_hash': '3beeed44291c4a55cfa346f4ad59fff...,CD11b,Integrin Alpha M (ITGAM); the alpha subunit of...
1,CD20,CD20,cd_marker,[{'row_hash': 'ce7741fc0da0adb15b4a0fb5fa947ae...,CD20,a type III transmembrane protein found on B ce...
2,CD3D,CD3d,cd_marker,[{'row_hash': '09c1942efdc37d825b122de9bf6a073...,CD3d,T-cell surface glycoprotein CD3 delta chain
3,CD45,CD45,cd_marker,[{'row_hash': '8250af52edfe0cd816c7f8bf324c600...,CD45,"leucocyte common antigen, a type I transmembra..."
4,CD4,CD4,cd_marker,[{'row_hash': 'f9c99c0b2e7cae7a72cc82d98ddaa81...,CD4,"a co-receptor for MHC Class II (with TCR, T-ce..."


In [54]:
# counnt unique cd_name matches
unique_cd_matches = cd_df['cd_name'].nunique()
print(f"Number of unique CD names matched: {unique_cd_matches}")

Number of unique CD names matched: 72


In [55]:
# was anyhing not matched
unmatched_cd = cd_df[cd_df['cd_name'].isna()]
print(f"Number of unmatched CD markers: {len(unmatched_cd)}")

unmatched_cd.head(20)

Number of unmatched CD markers: 8


,key_0,extracted_marker,marker_type,entries,cd_name,description
6,CD8,CD8,cd_marker,[{'row_hash': '258d9af17b6636cfa5bdb95410d6769...,NaN,NaN
7,CD3,CD3,cd_marker,[{'row_hash': '60263c018361ff0c816917fcf506a18...,NaN,NaN
19,CD45RO,CD45RO,cd_marker,[{'row_hash': 'cf5880d1297806a35b6ac6f1ee09adf...,NaN,NaN
27,CD45RA,CD45RA,cd_marker,[{'row_hash': 'dbeaf84c27811a6d4d6873b3644bc84...,NaN,NaN
54,CD16,CD16,cd_marker,[{'row_hash': '77952622f937826d7f10da1c3043824...,NaN,NaN
59,CD66,CD66,cd_marker,[{'row_hash': '577ee05a0b2562a4c163d1d03fab02a...,NaN,NaN
67,CD45R,CD45R,cd_marker,[{'row_hash': 'eca18b0387981c7f6f5272859009196...,NaN,NaN
73,CD40LG,CD40LG,cd_marker,[{'row_hash': '790a09edeb876bd7739986a1d6a0dd6...,NaN,NaN


In [62]:
# find extracted markers that are highly similar to each other
# use a fuzzy matching approach

similar_markers = []
marker_names = [entry['extracted_marker'] for entry in classified_marker_list]
from fuzzywuzzy import fuzz
from fuzzywuzzy import process

for i, marker in enumerate(marker_names):
    matches = process.extract(marker, marker_names, scorer=fuzz.token_sort_ratio)
    for match, score in matches:
        if score >= 90 and match != marker:
            similar_markers.append((marker, match, score))

print("Similar Markers (Score >= 90) ranked by similarity:")
similar_markers = sorted(similar_markers, key=lambda x: x[2], reverse=True)
for marker1, marker2, score in similar_markers:
    print(f"{marker1} <--> {marker2} (Score: {score})")

Similar Markers (Score >= 90) ranked by similarity:
HLA class I <--> HLA Class II (Score: 96)
HLA Class II <--> HLA class I (Score: 96)
beta3-integrin <--> beta-integrin (Score: 96)
beta-integrin <--> beta3-integrin (Score: 96)
G1/S-specific cyclin-E1 <--> G1/S-specific cyclin-D1 (Score: 96)
Cytokeratin 17 <--> Cytokeratin 7 (Score: 96)
G1/S-specific cyclin-D1 <--> G1/S-specific cyclin-E1 (Score: 96)
Cytokeratin 7 <--> Cytokeratin 17 (Score: 96)
collagen I <--> Collagen IV (Score: 95)
Collagen IV <--> collagen I (Score: 95)
Phospho-40S ribosomal protein S6 <--> Phospho Ribosomal Protein S6 (Score: 93)
Cytokeratin 14 <--> Cytokeratin 19 (Score: 93)
Cytokeratin 14 <--> Cytokeratin 17 (Score: 93)
Cytokeratin 19 <--> Cytokeratin 14 (Score: 93)
Cytokeratin 19 <--> Cytokeratin 17 (Score: 93)
Cytokeratin 17 <--> Cytokeratin 14 (Score: 93)
Cytokeratin 17 <--> Cytokeratin 19 (Score: 93)
Phospho Ribosomal Protein S6 <--> Phospho-40S ribosomal protein S6 (Score: 93)
Beta-Catenin <--> Catenin beta